# 06 -- Conversational Interface
**ITAI 2373 -- Final Project | Leroy Brown**

This notebook builds and demonstrates the NewsBot 2.0 Gradio-based conversational interface.
All seven intents (classify, summarize, search, translate, stats, topics, help) are exercised
with live examples before the full Gradio UI is launched.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import kagglehub

from config.settings import DATASET_SIZE, RANDOM_STATE, SEMANTIC_INDEX_SIZE
from src.data_processing.preprocessor import preprocess_text
from src.data_processing.data_loader import load_bbc_dataset
from src.data_processing.data_validator import validate_dataframe, clean_dataframe
from src.analysis.classifier import NewsClassifier
from src.analysis.sentiment_analyzer import score_text
from src.language_models.summarizer import summarize_with_stats
from src.language_models.embeddings import SemanticSearchIndex
from src.multilingual.translator import translate_and_detect
from src.conversation.query_processor import QueryProcessor
from src.conversation.intent_classifier import classify_intent, extract_content
from src.conversation.response_generator import HELP_TEXT

print("All imports OK.")

## 1 -- Load and Prepare the Pipeline Components

We reuse the same fitted objects from earlier notebooks.

In [ ]:
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")
df_raw = load_bbc_dataset(path)

report = validate_dataframe(df_raw)
df = clean_dataframe(df_raw).sample(min(DATASET_SIZE, len(df_raw)), random_state=RANDOM_STATE).reset_index(drop=True)
df["clean_text"] = df["text"].apply(preprocess_text)

print(f"Dataset: {len(df):,} articles | categories: {df['category'].nunique()}")
print(df["category"].value_counts().to_string())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), min_df=2)
X_tfidf = vectorizer.fit_transform(df["clean_text"])

clf = NewsClassifier(vectorizer=vectorizer)
results = clf.train_test_evaluate(X_tfidf, df["category"])
print(f"Classifier accuracy: {results['accuracy']:.1%}")

In [ ]:
search_index = SemanticSearchIndex()
search_index.build(df)
print(f"Search index built: {len(search_index.index)} articles indexed.")

In [ ]:
from src.analysis.sentiment_analyzer import score_text as _score

sentiments = df["text"].apply(_score)
df["compound"] = sentiments.apply(lambda s: s["sent_compound"])
df["word_count"] = df["text"].apply(lambda t: len(t.split()))

stats_cache = {
    "total_articles":  len(df),
    "accuracy":        results["accuracy"],
    "category_counts": df["category"].value_counts().to_dict(),
    "avg_sentiment":   df.groupby("category")["compound"].mean().to_dict(),
    "avg_words":       df.groupby("category")["word_count"].mean().round(0).astype(int).to_dict(),
    "top_topics":      {cat: ["news", "government", "market", "people", "global"]
                        for cat in df["category"].unique()},
}
print("Stats cache built.")

In [ ]:
qp = QueryProcessor(
    classifier    = clf,
    vectorizer    = vectorizer,
    preprocessor  = preprocess_text,
    summarizer_fn = summarize_with_stats,
    search_index  = search_index,
    translator_fn = translate_and_detect,
    sentiment_fn  = score_text,
    stats_cache   = stats_cache,
)
print("QueryProcessor ready.")

## 2 -- Intent Classification Demonstration

The rule-based intent classifier maps natural language to one of seven intents.

In [ ]:
test_queries = [
    ("classify: The government announced a new budget plan today.", "classify"),
    ("summarize: Arsenal beat Chelsea 3-1 in a thrilling Premier League match.", "summarize"),
    ("search: artificial intelligence technology companies", "search"),
    ("translate: El presidente firmo un nuevo acuerdo comercial.", "translate"),
    ("How many articles are in the dataset?", "stats"),
    ("What topics are covered in technology news?", "topics"),
    ("What can you do?", "help"),
    ("Tell me about the latest football scores", "fallback"),
]

print(f"{'Query':<65} {'Expected':<12} {'Got':<12} {'OK'}")
print("-" * 105)
ok = 0
for query, expected in test_queries:
    got = classify_intent(query)
    match = "OK" if got == expected else "FAIL"
    if got == expected:
        ok += 1
    print(f"{query[:63]:<65} {expected:<12} {got:<12} {match}")
print(f"\n{ok}/{len(test_queries)} intents correctly classified")

## 3 -- End-to-End Intent Responses

Each intent is exercised through the full `QueryProcessor.process()` pipeline.

In [ ]:
print("=" * 60)
print("INTENT: help")
print("=" * 60)
print(qp.process("help"))

In [ ]:
print("=" * 60)
print("INTENT: stats")
print("=" * 60)
print(qp.process("show me the stats"))

In [ ]:
print("=" * 60)
print("INTENT: topics")
print("=" * 60)
print(qp.process("what are the top topics?"))

In [ ]:
article_snippet = df[df["category"] == "politics"]["text"].iloc[0][:500]
print("=" * 60)
print("INTENT: classify")
print("=" * 60)
response = qp.process(f"classify: {article_snippet}")
print(response)

In [ ]:
sport_article = df[df["category"] == "sport"]["text"].iloc[0]
print("=" * 60)
print("INTENT: summarize")
print("=" * 60)
response = qp.process(f"summarize: {sport_article}")
print(response)

In [ ]:
print("=" * 60)
print("INTENT: search")
print("=" * 60)
print(qp.process("search: artificial intelligence in business"))

In [ ]:
print("=" * 60)
print("INTENT: translate")
print("=" * 60)
print(qp.process("translate: El presidente firmo un nuevo acuerdo comercial con Europa."))

In [ ]:
print("=" * 60)
print("INTENT: fallback")
print("=" * 60)
print(qp.process("Tell me something interesting about football"))

## 4 -- Multi-Turn Conversation Simulation

Simulates a realistic session with multiple sequential queries.

In [ ]:
conversation = [
    "help",
    "stats",
    "search: technology companies stock market",
    f"classify: {df[df['category']=='tech']['text'].iloc[0][:400]}",
    "translate: Die Regierung plant neue Wirtschaftsreformen fuer das naechste Jahr.",
    "what topics appear in entertainment?",
]

history = []
print("-- NewsBot 2.0 Conversation Simulation --\n")
for i, user_msg in enumerate(conversation, 1):
    print(f"User [{i}]: {user_msg[:80]}{'...' if len(user_msg)>80 else ''}")
    response = qp.process(user_msg, history=history)
    history.append((user_msg, response))
    preview = response[:200].replace("\n", " ")
    print(f"Bot  [{i}]: {preview}...\n")
print(f"Conversation complete -- {len(history)} turns.")

## 5 -- Gradio Conversational UI

Launches the interactive chatbot. In Colab a public share link is generated automatically.

In [ ]:
import gradio as gr

def chat(message, history):
    return qp.process(message, history)

demo = gr.ChatInterface(
    fn          = chat,
    title       = "NewsBot 2.0 -- AI News Intelligence Engine",
    description = (
        "ITAI 2373 Final Project | Leroy Brown\n\n"
        "Commands: **classify:** | **summarize:** | **search:** | "
        "**translate:** | **stats** | **topics** | **help**"
    ),
    examples    = [
        "help",
        "stats",
        "search: climate change renewable energy",
        "topics",
    ],
    theme       = gr.themes.Soft(),
)

demo.launch(share=True, debug=False)

## 6 -- Summary

| Feature | Implementation |
|---------|---------------|
| Intent classification | Rule-based keyword matching (7 intents) |
| Classify intent | TF-IDF + Logistic Regression + VADER |
| Summarize intent | DistilBART abstractive summarization |
| Search intent | Sentence-BERT semantic similarity |
| Translate intent | langdetect + Google Translate |
| Stats intent | Pre-computed dataset analytics |
| Topics intent | LDA/NMF top keywords per category |
| UI framework | Gradio Blocks ChatInterface |

The conversational interface successfully routes all seven intents to the appropriate NLP backend.